# 🚢 Machine Learning Workshop — Who Survived the Titanic?
### Beginner's Guide to Classification with Python

---

## 🎯 What Will You Learn Today?

By the end of this notebook, you will know how to:
- **Explore** a real-world dataset and understand what's inside
- **Clean** messy data (missing values, text columns, etc.)
- **Train** a Machine Learning model to make predictions
- **Evaluate** how good your model is
- **Make predictions** on new, unseen passengers

---

## 🧊 The Story

On April 15, 1912, the RMS Titanic sank after hitting an iceberg.  
Of the **2,224 passengers and crew**, only **710 survived**.

We have a dataset with information about 891 of those passengers —  
their age, ticket class, gender, and more — **and whether they survived**.

Our goal: **Train a model that can look at a passenger's information and predict whether they survived.**

This type of problem is called **Classification** — the model predicts one of two categories:
- `0` = Did not survive ❌
- `1` = Survived ✅

---

> 💡 **No prior ML experience required!** Every step is explained in plain language.

---
## STEP 1 — Import Libraries

Think of libraries as **toolboxes**. Before you start a job, you grab the tools you need.

- `pandas` → for working with tables of data
- `matplotlib` / `seaborn` → for making charts and visualizations
- `sklearn` → the main Machine Learning toolbox

In [ ]:
# ── Toolboxes ─────────────────────────────────────────────────────────────────
import pandas as pd               # for working with tables
import numpy as np                # for math
import matplotlib.pyplot as plt   # for charts
import seaborn as sns             # for prettier charts
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning tools ────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

# ── Chart settings ────────────────────────────────────────────────────────────
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid')

print('✅ All tools loaded successfully!')

---
## STEP 2 — Load the Data

We have two files:
- `train.csv` → 891 passengers we **know** survived or not. We'll use this to **teach** the model.
- `test.csv`  → 418 passengers where we **don't know** the outcome. We'll ask the model to **predict** it.

In [ ]:
# Load the data files
# If using Google Colab, upload the files first using the Files panel on the left,
# or mount Google Drive and point the path to where you stored them.

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(f'Training set  : {train.shape[0]} passengers, {train.shape[1]} columns')
print(f'Test set      : {test.shape[0]}  passengers, {test.shape[1]} columns')

---
## STEP 3 — Explore the Data
### "What information do we have about each passenger?"

Before training any model, we need to **understand our data**.  
This phase is called **EDA (Exploratory Data Analysis)**.

Here is what each column means:

| Column | Meaning |
|---|---|
| `PassengerId` | Just a number, not useful for prediction |
| `Survived` | **Our target: 0 = No, 1 = Yes** |
| `Pclass` | Ticket class (1 = 1st/rich, 2 = 2nd, 3 = 3rd/poor) |
| `Name` | Full name |
| `Sex` | male / female |
| `Age` | Age in years |
| `SibSp` | # of siblings or spouse on board |
| `Parch` | # of parents or children on board |
| `Ticket` | Ticket number |
| `Fare` | Price paid for the ticket |
| `Cabin` | Cabin number (many are missing) |
| `Embarked` | Port of embarkation (C=Cherbourg, Q=Queenstown, S=Southampton) |

In [ ]:
# Show the first 5 rows — like opening the spreadsheet
print('── First 5 rows of training data ──────────────────────────────────────')
train.head()

In [ ]:
# Overall statistics for each column
print('── Statistical Summary ─────────────────────────────────────────────────')
train.describe().round(2)

In [ ]:
# How many missing values in each column?
print('── Missing Values ──────────────────────────────────────────────────────')
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(1)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

print('\n💡 Age is missing for ~20% of passengers.')
print('   Cabin is missing for ~77% — too many gaps to use reliably.')
print('   Embarked is missing for only 2 passengers — easy to fix.')

In [ ]:
# ── How many survived vs. did not survive? ────────────────────────────────────
counts = train['Survived'].value_counts()
labels = ['Did Not Survive ❌', 'Survived ✅']
colors = ['#e74c3c', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=13, fontweight='bold')
axes[0].set_title('How many survived?', fontweight='bold', fontsize=14)
axes[0].set_ylabel('Number of passengers')
axes[0].set_ylim(0, 600)

# Pie chart
axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Survival Rate', fontweight='bold', fontsize=14)

plt.suptitle('Overall Survival — Titanic (Training Data)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

survival_rate = counts[1] / counts.sum() * 100
print(f'\n📊 {survival_rate:.1f}% of passengers in our training data survived.')

In [ ]:
# ── Survival by Gender ────────────────────────────────────────────────────────
# This is one of the most famous patterns in the Titanic data

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count chart
sex_survival = train.groupby(['Sex', 'Survived']).size().unstack()
sex_survival.plot(kind='bar', color=['#e74c3c', '#2ecc71'], edgecolor='white',
                  ax=axes[0], rot=0)
axes[0].set_title('Survival Count by Gender', fontweight='bold')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Number of passengers')
axes[0].legend(['Did Not Survive', 'Survived'], title='Outcome')

# Rate chart
sex_rate = train.groupby('Sex')['Survived'].mean() * 100
bars = axes[1].bar(sex_rate.index, sex_rate.values, color=['#3498db', '#e91e8c'],
                   edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, sex_rate.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 1,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=13)
axes[1].set_title('Survival Rate by Gender', fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_ylim(0, 100)

plt.suptitle('"Women and children first" — does the data confirm it?', fontsize=13, style='italic')
plt.tight_layout()
plt.show()

print(f'\n👩 Female survival rate : {sex_rate["female"]:.1f}%')
print(f'👨 Male survival rate   : {sex_rate["male"]:.1f}%')
print('\n💡 Gender is probably going to be one of the most important features for our model!')

In [ ]:
# ── Survival by Ticket Class ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_labels = {1: '1st Class\n(Wealthy)', 2: '2nd Class\n(Middle)', 3: '3rd Class\n(Poor)'}
pclass_rate  = train.groupby('Pclass')['Survived'].mean() * 100
pclass_count = train.groupby(['Pclass', 'Survived']).size().unstack()

pclass_count.plot(kind='bar', color=['#e74c3c', '#2ecc71'],
                  edgecolor='white', ax=axes[0], rot=0)
axes[0].set_title('Survival Count by Ticket Class', fontweight='bold')
axes[0].set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
axes[0].set_ylabel('Number of passengers')
axes[0].legend(['Did Not Survive', 'Survived'])

bar_colors = ['#f39c12', '#95a5a6', '#7f8c8d']
bars = axes[1].bar(['1st Class', '2nd Class', '3rd Class'], pclass_rate.values,
                   color=bar_colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, pclass_rate.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 1,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=13)
axes[1].set_title('Survival Rate by Ticket Class', fontweight='bold')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_ylim(0, 100)

plt.suptitle('Did wealth affect survival chances?', fontsize=13, style='italic')
plt.tight_layout()
plt.show()

for cls, rate in pclass_rate.items():
    print(f'  Class {cls}: {rate:.1f}% survived')

In [ ]:
# ── Survival by Age ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram: age distribution of survivors vs non-survivors
train[train['Survived'] == 0]['Age'].dropna().plot(
    kind='hist', bins=30, alpha=0.6, color='#e74c3c', label='Did Not Survive', ax=axes[0])
train[train['Survived'] == 1]['Age'].dropna().plot(
    kind='hist', bins=30, alpha=0.6, color='#2ecc71', label='Survived', ax=axes[0])
axes[0].set_title('Age Distribution: Survivors vs Non-Survivors', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Number of passengers')
axes[0].legend()
axes[0].axvline(10, color='blue', linestyle='--', linewidth=1.5, label='Age 10 (children)')

# Survival rate by age group
train['AgeGroup'] = pd.cut(train['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Child\n(0-12)', 'Teen\n(13-18)', 'Adult\n(19-35)', 'Middle\n(36-60)', 'Senior\n(60+)'])
age_rate = train.groupby('AgeGroup', observed=True)['Survived'].mean() * 100
bars = axes[1].bar(age_rate.index, age_rate.values, color='#3498db', edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, age_rate.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 1,
                 f'{val:.0f}%', ha='center', fontweight='bold')
axes[1].set_title('Survival Rate by Age Group', fontweight='bold')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_ylim(0, 80)

plt.tight_layout()
plt.show()
print('💡 Children had the highest survival rate — "women and children first" confirmed!')

In [ ]:
# ── Combined view: Gender + Class ────────────────────────────────────────────
plt.figure(figsize=(10, 6))
pivot = train.pivot_table(values='Survived', index='Pclass', columns='Sex', aggfunc='mean') * 100
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
            linewidths=0.5, cbar_kws={'label': 'Survival Rate (%)'},
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title('Survival Rate (%) by Class and Gender\n(Green = high survival, Red = low survival)',
          fontweight='bold', fontsize=13)
plt.xlabel('Gender')
plt.ylabel('Ticket Class')
plt.yticks(ticks=[0.5, 1.5, 2.5], labels=['1st Class', '2nd Class', '3rd Class'], rotation=0)
plt.tight_layout()
plt.show()

print('💡 The most dangerous position: 3rd Class Male — only ~13% survived!')
print('   The safest position: 1st Class Female — ~97% survived!')

---
## STEP 4 — Clean the Data

Real-world data is always messy. Before training a model, we need to:
1. **Fill in missing values** — ML models can't handle empty cells
2. **Convert text to numbers** — ML models only understand numbers
3. **Drop columns we don't need**

> 💡 This process is called **Data Preprocessing** — it's often the most important step in ML!

In [ ]:
def clean_data(df):
    """
    Apply all cleaning steps to a dataset.
    We write this as a function so we can apply the
    EXACT same steps to both train and test data.
    """
    data = df.copy()   # don't modify the original

    # ── 1. Fill missing Age with the median age ────────────────────────────────
    # Median is better than mean here because age has outliers
    median_age = data['Age'].median()
    data['Age'] = data['Age'].fillna(median_age)
    print(f'  ✅ Missing Age filled with median: {median_age:.1f} years')

    # ── 2. Fill missing Embarked with the most common port ────────────────────
    most_common_port = data['Embarked'].mode()[0]
    data['Embarked'] = data['Embarked'].fillna(most_common_port)
    print(f'  ✅ Missing Embarked filled with most common: "{most_common_port}"')

    # ── 3. Fill missing Fare (only appears in test set) ────────────────────────
    data['Fare'] = data['Fare'].fillna(data['Fare'].median())
    print(f'  ✅ Missing Fare filled with median')

    # ── 4. Convert Sex to a number (male=0, female=1) ─────────────────────────
    # ML models need numbers, not words
    data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})
    print(f'  ✅ Sex converted → male=0, female=1')

    # ── 5. Convert Embarked to numbers ────────────────────────────────────────
    data['Embarked'] = data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
    print(f'  ✅ Embarked converted → S=0, C=1, Q=2')

    # ── 6. Create a new feature: Family Size ──────────────────────────────────
    # SibSp + Parch + 1 (the passenger themselves)
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    print(f'  ✅ New feature created: FamilySize = SibSp + Parch + 1')

    # ── 7. Create a feature: Is the passenger alone? ──────────────────────────
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    print(f'  ✅ New feature created: IsAlone (1=alone, 0=with family)')

    # ── 8. Drop columns we won't use ──────────────────────────────────────────
    # Name, Ticket, Cabin: too unique or too many missing values
    cols_to_drop = ['Name', 'Ticket', 'Cabin', 'PassengerId']
    data = data.drop(columns=[c for c in cols_to_drop if c in data.columns])
    print(f'  ✅ Dropped columns: {cols_to_drop}')

    return data


print('── Cleaning TRAINING data ───────────────────────────────────────────────')
train_clean = clean_data(train)

print('\n── Cleaning TEST data ───────────────────────────────────────────────────')
test_clean = clean_data(test)

print('\n✅ Cleaned training data preview:')
train_clean.head()

In [ ]:
# Verify: no more missing values!
print('── Missing values after cleaning ────────────────────────────────────────')
print(train_clean.isnull().sum())
print('\n🎉 Zero missing values — the data is ready for ML!')

---
## STEP 5 — Prepare Features and Target

In Machine Learning, we separate the data into:
- **X (Features)** — the information we use to make the prediction (inputs)
- **y (Target)** — what we're trying to predict (output)

Then we split into **Training** and **Validation** sets:
- **Training set** → the model learns from this
- **Validation set** → we test the model on data it has NEVER seen before

> 🎓 **Analogy:** It's like studying for an exam. You practice with sample questions (training), then take the real exam with new questions (validation). If you just memorized the sample answers without understanding, you'll fail the real exam — that's called **overfitting**.

In [ ]:
from sklearn.model_selection import train_test_split

# ── Define Features (X) and Target (y) ───────────────────────────────────────
FEATURES = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch',
            'Fare', 'Embarked', 'FamilySize', 'IsAlone']

TARGET   = 'Survived'

X = train_clean[FEATURES]
y = train_clean[TARGET]

# ── Split: 80% for training, 20% for validation ───────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,    # 20% goes to validation
    random_state=42,   # fixed seed = same split every time
    stratify=y         # make sure both splits have similar survival rates
)

print(f'Total passengers  : {len(X)}')
print(f'Training samples  : {len(X_train)} passengers  → model LEARNS from these')
print(f'Validation samples: {len(X_val)}  passengers  → model is TESTED on these (unseen)')
print(f'\nFeatures used: {FEATURES}')

---
## STEP 6 — Train Model 1: Logistic Regression

Our first model is **Logistic Regression**.

Despite its name, it's a **classification** model (not regression!).

**How it works:**  
It looks at each feature and assigns it a weight — like a voting system. Features that strongly predict survival get bigger weights. Then it combines everything to output a **probability between 0 and 1**.

- Probability > 0.5 → Predict **Survived** ✅
- Probability ≤ 0.5 → Predict **Did not survive** ❌

> 💡 Think of it as a **weighted checklist** the model fills out for each passenger.

In [ ]:
# ── Train Logistic Regression ─────────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)   # ← this is where learning happens!

# ── Make predictions ──────────────────────────────────────────────────────────
lr_pred = lr.predict(X_val)

# ── Accuracy ──────────────────────────────────────────────────────────────────
lr_acc = accuracy_score(y_val, lr_pred)
print(f'✅ Logistic Regression trained!')
print(f'   Accuracy on validation set: {lr_acc*100:.1f}%')
print(f'\n   Interpretation: The model correctly predicted {lr_acc*100:.1f}%')
print(f'   of the {len(y_val)} passengers it had never seen before.')

In [ ]:
# ── See what the model learned: Feature Weights ───────────────────────────────
# Positive weight → feature increases survival probability
# Negative weight → feature decreases survival probability

weights_df = pd.DataFrame({
    'Feature': FEATURES,
    'Weight':  lr.coef_[0]
}).sort_values('Weight')

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if w < 0 else '#2ecc71' for w in weights_df['Weight']]
plt.barh(weights_df['Feature'], weights_df['Weight'], color=colors)
plt.axvline(0, color='black', linewidth=1)
plt.title('Logistic Regression — What Did the Model Learn?\n'
          'Green = increases survival chance | Red = decreases survival chance',
          fontweight='bold')
plt.xlabel('Weight (importance)')
plt.tight_layout()
plt.show()

print('💡 Sex (female=1) has a large positive weight → being female increases survival chance')
print('   Pclass has a negative weight → higher class number (3rd) decreases survival chance')

---
## STEP 7 — Train Model 2: Decision Tree

Our second model is a **Decision Tree**.

**How it works:**  
It learns a series of **yes/no questions** about the passenger and follows a path to a prediction — like a flowchart.

For example:
> *Is the passenger female? → Yes → Did they pay more than $30? → Yes → Predict: Survived!*

> 💡 This is the most **visual and intuitive** ML model — you can literally draw it out and follow the logic!

In [ ]:
# ── Train Decision Tree ───────────────────────────────────────────────────────
# max_depth=4 limits the tree so it doesn't get too complex
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

# ── Make predictions ──────────────────────────────────────────────────────────
dt_pred = dt.predict(X_val)

# ── Accuracy ──────────────────────────────────────────────────────────────────
dt_acc = accuracy_score(y_val, dt_pred)
print(f'✅ Decision Tree trained!')
print(f'   Accuracy on validation set: {dt_acc*100:.1f}%')

In [ ]:
# ── Visualize the Decision Tree ───────────────────────────────────────────────
# This is one of the coolest features of Decision Trees!

plt.figure(figsize=(22, 10))
plot_tree(
    dt,
    feature_names=FEATURES,
    class_names=['Did Not Survive', 'Survived'],
    filled=True,
    rounded=True,
    fontsize=10,
    impurity=False
)
plt.title('Decision Tree — The Model\'s Flowchart\n'
          '(Blue = leans toward Survived | Orange = leans toward Did Not Survive)',
          fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

print('💡 You can follow any path from the top (root) to the bottom (leaf) to see how the model decides!')
print('   Each box shows: the question asked, how many samples reached that point, and the predicted class.')

In [ ]:
# ── Feature Importance from Decision Tree ────────────────────────────────────
fi_df = pd.DataFrame({
    'Feature':    FEATURES,
    'Importance': dt.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(fi_df['Feature'], fi_df['Importance'], color='#3498db', edgecolor='white')
plt.title('Decision Tree — Feature Importance\n'
          '(How much each feature was used to make decisions)',
          fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

top_feature = fi_df.iloc[-1]['Feature']
print(f'🏆 Most important feature: {top_feature}')

---
## STEP 8 — Evaluate the Models

**Accuracy alone isn't enough!**  

Imagine a model that always says "Did Not Survive" — it would be right 61% of the time just because 61% of passengers didn't survive. But it's a terrible, useless model.

We need better metrics:

| Metric | What it measures |
|---|---|
| **Accuracy** | % of all predictions that were correct |
| **Precision** | Of those predicted as Survived, how many actually did? |
| **Recall** | Of those who actually survived, how many did we catch? |
| **Confusion Matrix** | Full breakdown of correct and incorrect predictions |

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, name in zip(
    axes,
    [lr_pred, dt_pred],
    ['Logistic Regression', 'Decision Tree']
):
    cm = confusion_matrix(y_val, preds)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Did Not Survive ❌', 'Survived ✅']
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_val, preds)
    ax.set_title(f'{name}\nAccuracy: {acc*100:.1f}%', fontweight='bold', fontsize=13)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — How Many Did Each Model Get Right?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('📖 HOW TO READ A CONFUSION MATRIX:')
print('   Top-left     → True Negatives  (correctly predicted: did not survive)')
print('   Bottom-right → True Positives  (correctly predicted: survived)')
print('   Top-right    → False Positives (predicted survived, but actually did not)')
print('   Bottom-left  → False Negatives (predicted did not survive, but actually did)')

In [ ]:
# ── Full Classification Report ────────────────────────────────────────────────
print('═'*55)
print('       LOGISTIC REGRESSION — Full Report')
print('═'*55)
print(classification_report(y_val, lr_pred,
      target_names=['Did Not Survive', 'Survived']))

print('═'*55)
print('       DECISION TREE — Full Report')
print('═'*55)
print(classification_report(y_val, dt_pred,
      target_names=['Did Not Survive', 'Survived']))

In [ ]:
# ── Cross-Validation: a more reliable accuracy estimate ───────────────────────
# Instead of one train/val split, we do 5 different splits and average the results
# This gives a fairer picture of how the model will perform on new data

print('⏳ Running 5-fold Cross-Validation...')

cv_lr = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5, scoring='accuracy')
cv_dt = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=42), X, y, cv=5, scoring='accuracy')

print(f'\n  Logistic Regression → {cv_lr.mean()*100:.1f}% ± {cv_lr.std()*100:.1f}%  (across 5 folds)')
print(f'  Decision Tree       → {cv_dt.mean()*100:.1f}% ± {cv_dt.std()*100:.1f}%  (across 5 folds)')

# Comparison bar chart
plt.figure(figsize=(8, 5))
models_names = ['Logistic\nRegression', 'Decision\nTree']
means = [cv_lr.mean()*100, cv_dt.mean()*100]
stds  = [cv_lr.std()*100,  cv_dt.std()*100]
colors = ['#3498db', '#e67e22']
bars = plt.bar(models_names, means, color=colors, edgecolor='white',
               linewidth=1.5, yerr=stds, capsize=8, width=0.5)
for bar, mean in zip(bars, means):
    plt.text(bar.get_x() + bar.get_width()/2, mean + 0.5,
             f'{mean:.1f}%', ha='center', fontweight='bold', fontsize=13)
plt.title('Model Comparison — Cross-Validation Accuracy\n(Error bars show variability)', fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.ylim(60, 90)
plt.tight_layout()
plt.show()

---
## STEP 9 — Make Predictions on New Passengers

Now the fun part! Let's use our trained models to predict whether **made-up passengers** would have survived.  

We'll also run predictions on the real **test.csv** file.

In [ ]:
# ── Predict for made-up passengers ───────────────────────────────────────────
# Change these values and see how the prediction changes!

made_up_passengers = pd.DataFrame([
    # Pclass, Sex, Age,  SibSp, Parch, Fare,   Embarked, FamilySize, IsAlone
    {  'Pclass': 1, 'Sex': 1, 'Age': 30, 'SibSp': 0, 'Parch': 0,  # 1st class woman, alone
       'Fare': 100, 'Embarked': 0, 'FamilySize': 1, 'IsAlone': 1 },

    {  'Pclass': 3, 'Sex': 0, 'Age': 25, 'SibSp': 0, 'Parch': 0,  # 3rd class man, alone
       'Fare': 8,   'Embarked': 0, 'FamilySize': 1, 'IsAlone': 1 },

    {  'Pclass': 2, 'Sex': 0, 'Age': 8,  'SibSp': 1, 'Parch': 2,  # 2nd class boy, with family
       'Fare': 25,  'Embarked': 0, 'FamilySize': 4, 'IsAlone': 0 },

    {  'Pclass': 1, 'Sex': 0, 'Age': 45, 'SibSp': 1, 'Parch': 0,  # 1st class rich man, with wife
       'Fare': 200, 'Embarked': 1, 'FamilySize': 2, 'IsAlone': 0 },

    {  'Pclass': 3, 'Sex': 1, 'Age': 20, 'SibSp': 0, 'Parch': 0,  # 3rd class young woman, alone
       'Fare': 7.5, 'Embarked': 2, 'FamilySize': 1, 'IsAlone': 1 },
])

descriptions = [
    '1st class woman, 30 yrs, alone',
    '3rd class man,   25 yrs, alone',
    '2nd class boy,    8 yrs, with family',
    '1st class man,   45 yrs, with wife',
    '3rd class woman, 20 yrs, alone'
]

lr_preds_new = lr.predict(made_up_passengers[FEATURES])
lr_prob_new  = lr.predict_proba(made_up_passengers[FEATURES])[:, 1]  # probability of survival

print('═'*65)
print('   PREDICTIONS FOR MADE-UP PASSENGERS (Logistic Regression)')
print('═'*65)
for desc, pred, prob in zip(descriptions, lr_preds_new, lr_prob_new):
    outcome = '✅ SURVIVED' if pred == 1 else '❌ DID NOT SURVIVE'
    print(f'  {desc:<40} → {outcome}  ({prob*100:.0f}% chance)')
print('═'*65)

In [ ]:
# ── Run predictions on the real test.csv ─────────────────────────────────────
test_ids   = test['PassengerId'].copy()
X_test_ml  = test_clean[FEATURES]

final_predictions = lr.predict(X_test_ml)

# Build submission file
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived':    final_predictions
})

submission.to_csv('titanic_predictions.csv', index=False)

print('✅ Predictions saved to titanic_predictions.csv')
print(f'\nOf {len(final_predictions)} test passengers:')
print(f'  {(final_predictions == 1).sum()} predicted to SURVIVE   ({(final_predictions==1).mean()*100:.1f}%)')
print(f'  {(final_predictions == 0).sum()} predicted to NOT survive ({(final_predictions==0).mean()*100:.1f}%)')

print('\nFirst 10 predictions:')
print(submission.head(10).to_string(index=False))

---
## STEP 10 — 🧠 Try It Yourself!

Now it's your turn to experiment. Try changing things and see what happens!

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║         🧪 EXPERIMENT 1: Change the passenger details        ║
# ║   Modify the values below and run the cell to see if        ║
# ║   the prediction changes!                                   ║
# ╚══════════════════════════════════════════════════════════════╝

my_passenger = pd.DataFrame([{
    'Pclass':     3,       # ← Try: 1, 2, or 3
    'Sex':        0,       # ← Try: 0 = male, 1 = female
    'Age':        25,      # ← Try any age
    'SibSp':      0,       # ← siblings/spouses on board
    'Parch':      0,       # ← parents/children on board
    'Fare':       8.0,     # ← Try: 5 (poor), 50 (middle), 200 (rich)
    'Embarked':   0,       # ← 0=Southampton, 1=Cherbourg, 2=Queenstown
    'FamilySize': 1,       # ← SibSp + Parch + 1
    'IsAlone':    1,       # ← 1=alone, 0=with family
}])

prediction  = lr.predict(my_passenger[FEATURES])[0]
probability = lr.predict_proba(my_passenger[FEATURES])[0][1]

print('──────────────────────────────────────────')
if prediction == 1:
    print(f'  PREDICTION: ✅ SURVIVED')
else:
    print(f'  PREDICTION: ❌ DID NOT SURVIVE')
print(f'  Survival probability: {probability*100:.1f}%')
print('──────────────────────────────────────────')
print('\nTry changing Pclass from 3 → 1 and rerun. Does it change?')
print('Try changing Sex from 0 → 1. Does that make a big difference?')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   🧪 EXPERIMENT 2: What happens if we change max_depth?     ║
# ╚══════════════════════════════════════════════════════════════╝

depths = range(1, 12)
train_scores, val_scores = [], []

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    model.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, model.predict(X_train)) * 100)
    val_scores.append(accuracy_score(y_val,   model.predict(X_val))   * 100)

plt.figure(figsize=(10, 5))
plt.plot(depths, train_scores, 'o-', color='#3498db', label='Training Accuracy',   linewidth=2)
plt.plot(depths, val_scores,   's-', color='#e74c3c', label='Validation Accuracy', linewidth=2)
plt.title('Overfitting Demo — What happens as the tree gets deeper?', fontweight='bold')
plt.xlabel('Max Depth of Decision Tree')
plt.ylabel('Accuracy (%)')
plt.legend(fontsize=12)
plt.xticks(depths)
plt.tight_layout()
plt.show()

print('💡 NOTICE:')
print('   Training accuracy keeps going UP as depth increases — the model memorizes the training data.')
print('   Validation accuracy peaks then drops — the model stopped GENERALIZING and started MEMORIZING.')
print('   This is called OVERFITTING — the model is "too smart" about the training data but fails on new data.')

---
## 📋 Final Summary

### What you did in this workshop:

1. **Loaded** real passenger data from the Titanic disaster
2. **Explored** the data and found key patterns (gender, class, age matter!)
3. **Cleaned** the data: filled missing values, converted text to numbers
4. **Engineered** new features (FamilySize, IsAlone)
5. **Trained** two ML models: Logistic Regression and Decision Tree
6. **Evaluated** the models with Accuracy, Confusion Matrix, and Cross-Validation
7. **Predicted** survival for new unseen passengers
8. **Experimented** with overfitting and hyperparameters

---

### 💬 Discussion Questions

1. **Which model performed better — Logistic Regression or Decision Tree?** Is a higher accuracy always better?

2. **Look at the heatmap from Step 3.** A 1st-class woman had a 97% survival rate. A 3rd-class man had ~13%. What does this tell you about how society worked in 1912?

3. **What is overfitting?** Look at the Experiment 2 chart — at what depth does the Decision Tree start to overfit?

4. **Why can't we shuffle and randomly split a dataset before training?** *(For time-series data — what would go wrong?)*

5. **CHALLENGE:** Can you create a new feature? For example: `data['Child'] = (data['Age'] < 12).astype(int)`. Does it improve the accuracy?

6. **CHALLENGE:** Try training a `DecisionTreeClassifier` with `max_depth=None` (unlimited depth). What accuracy do you get on training vs. validation? What does this demonstrate?

---

*🚢 Workshop prepared for Mechatronics Engineering Students — Introduction to Machine Learning*